If its the first time you are running webscraping you will need to install the below libs. Just remove the #'s'

In [1]:
#pip intall fuzzywuzzy
#pip install BeautifulSoup
#pip install requests

In [2]:
from bs4 import BeautifulSoup
import requests, os, csv, urllib.parse, urllib3, time, random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import numpy as np
import pandas as pd

import re,os,glob, time,string
urllib3.disable_warnings()

import web_general
from IPython.display import Image

Create a blank dataframe to store the results

In [13]:
df = pd.DataFrame(columns=["Company_Name","Location","Revenue","Employees","Summary","Year_est","Industries"])
dict_={"Company_Name","Location","Revenue","Employees","Summary","Year_est","Industries"}

Enter the URL of the website you will like to webscrape

In [3]:
URL='https://www.dnb.com/business-directory/company-information.manufacturing.za.html?page=2'

#Dont change for this website
base_url="https://www.dnb.com/"

In [4]:
soup=web_general.parse_listing(web_general.buildSearchUrl(URL), web_general.get_proxies())
if not soup:print("Not found")
else: print("good to go")

good to go


![title](Overview.png)

We can now webscrape everything that we can see in the soup file.<br>
The information for each company is stored in a table set up. Where col-md-12 data represents the rows<br>
col-md-6 contains the name of the company + hyperlink<br>
col-md-4 contains the location<br>
col-md-2 contains the sales revenue<br>

In [7]:
soup;

![title](Table.png)

We want to find all the rows in the page. This gets stored in a list format. 

In [8]:
showing_per_page=soup.find_all("div", class_="col-md-12 data")
showing_per_page[0]#lets view the first row

In [10]:
get_hyperlink=showing_per_page[0].find('a').get('href')
get_hyperlink

'/business-directory/company-profiles.pick_n_pay_retailers_(pty)_ltd.2ea376bbff40d301510994a8ff13fe7b.html'

In [11]:
get_name=showing_per_page[0].find("div", class_="col-md-6").text.split()
name=" ".join(get_name)
name

'PICK N PAY RETAILERS (PTY) LTD'

In [12]:
get_location=showing_per_page[0].find("div", class_="col-md-4").text.split()
location=" ".join(get_location[1:])  #Remove country tag
location

'Kenilworth, Western Cape, South Africa'

In [13]:
get_revenue=showing_per_page[0].find("div", class_="col-md-2").text.split()
revenue=" ".join(get_revenue[-1]) #only pull revenue value
revenue

'$ 6 1 1 . 4 4 M'

There are 50 business names per a page. We will need to loop through then to get all of the info

In [14]:
len(soup.find_all("div", class_="col-md-12 data"))

50

We also want to get company information from the hyperlink

In [15]:
new_url=base_url+get_hyperlink
new_url

'https://www.dnb.com//business-directory/company-profiles.pick_n_pay_retailers_(pty)_ltd.2ea376bbff40d301510994a8ff13fe7b.html'

Lets create a new soup (object) file to extract this information

In [16]:
soup_2=web_general.parse_listing(web_general.buildSearchUrl(new_url), web_general.get_proxies())

![title](CS.png)

In [17]:
company_summary=soup_2.find_all("span", class_="company_summary")[0].text
company_summary

'PICK N PAY RETAILERS (PTY) LTD is located in KENILWORTH, Western Cape, South Africa and is part of the {{industry}} Industry. PICK N PAY RETAILERS (PTY) LTD has 80,000 total employees across all of its locations and generates $611.44 million in sales (USD). There are 22 companies in the PICK N PAY RETAILERS (PTY) LTD corporate family.\r\n\r\nD&B Hoovers provides sales leads and sales intelligence data on over 120 million companies like PICK N PAY RETAILERS (PTY) LTD around the world, including contacts, financials, and competitor information. To witness the full depth and breadth of our data and for industry leading sales intelligence tools, take D&B Hoovers for a test drive. Try D&B Hoovers Free'

To get the Industries. We can to extract all the tags

![title](PI.png)

In [18]:
industry_list=[]
for item in range(len(soup_2.find_all("span", class_="profile-industry-item"))):
    indus=soup_2.find_all("span", class_="profile-industry-item")[item].text.split()
    indus=" ".join(indus)
    industry_list.append(indus)
industry_list

['Direct Selling Establishments',
 'Specialty Food Stores',
 'Bakeries and Tortilla Manufacturing',
 'Clothing Stores',
 'Beer, Wine, and Liquor Stores',
 'Direct selling establishments',
 'Meat and fish markets',
 'Fruit and vegetable markets',
 'Retail bakeries',
 'Family clothing stores',
 'Liquor stores']

Lastly we want to pull the company snapshot

![title](Snap.png)

In [19]:
(soup_2.find_all("li", class_="empCon")[0].text.split())[-1]

'80,000'

In [20]:
(soup_2.find_all("li", class_="sales")[0].text.split())[-1]

'611.44'

In [21]:
(soup_2.find_all("li", class_="year_started")[0].text.split())[-1]

'1973'